# MAG-S3 в Google Colab

Блокнот несёт замороженные файлы MAG-S3 (протокол v0.4) и запускает их на всех процессорных ядрах среды.
Среда: «Среда выполнения → Сменить среду выполнения → TPU v5e-1». Сам TPU расчёт не использует, ему нужны 24 ядра
этой машины.

Порядок:
1. Ячейки 1 и 2 – подготовка: файлы со сверкой SHA-256, пакеты закреплённых версий, Google Диск.
2. Ячейка 3 – проверка окружения на калибровочных птицах 1001–1010 и 2001–3000 (около 15 минут).
   Пришли вывод или файл `MAG-S3/env_check.json` с Диска.
3. Ячейка 4 – слепая проверка конвейера на птицах 9301–9302 (около 15 минут). Она печатает только технические
   факты и удаляет свой выход.
4. Ячейка 5 – подтверждающий прогон. Запускать **только после заморозки и команды «запускай»**.

Всё, что нужно сохранить, пишется в папку `MAG-S3` на Google Диске.

In [ ]:
import os, subprocess, sys, time

ROOT, WORK, DRIVE = "/content/mag_s3", "/content/work", "/content/drive/MyDrive/MAG-S3"


def sh(cmd, cwd=ROOT, env=None):
    """Run a shell command, stream its output, return the exit code."""
    e = dict(os.environ, WORKERS=str(os.cpu_count()))
    e.update(env or {})
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

# 1. Файлы MAG-S3 и их SHA-256
import base64, hashlib, io, json, tarfile

FROZEN = {
 "mag_s3.py": "1df703af63a2af71a94385d33276ad39a422e9abaa040fdf4bcb618344e1f350",
 "analyze.py": "ecc6a15b77e4f4d1c07a4217439e0c97868b9a12971c9ada255f79840ef1d21f",
 "test_analyze.py": "7db7ee7c26c00b60e8ea902372c0cb24df9886b33d8aef3a5d798b00342ed272",
 "mag_s2.py": "2352aefe0fe5974732c318b365f4fb5477ffc47fb78b12748bca2515465829c4",
 "physics.json": "647bc0ccb1312991a1b772193cac39f4ced1481c591422c50659bb822a3100c5",
 "mag_s2_bird1.json": "0b603a65362e999a0c1da09ee0d6edc0e2864dcd690a2f277978d541943b2d86",
 "requirements.txt": "ed2d45295d87293d9cc367a7bd2740fa850a43b7d2d3778356fbc80e46d83980",
 "scripts/pipeline.sh": "3698071f4d1c0ed5d6e696c5779d2a41e70306bb6f5af9e93f7bb66db8e80247",
 "scripts/env_check.py": "094ca8e141e476d8cf4dbc0e0099f3b42549a9b21c8f04e5285f74d2b095c3c8"
}

PAYLOAD = """
...
fPh8+Hz4fPjc7/N/AMJT74QASAMA
"""

os.makedirs(ROOT, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:gz") as tar:
    tar.extractall(ROOT, filter="data")
bad = []
for name, want in FROZEN.items():
    got = hashlib.sha256(open(os.path.join(ROOT, name), "rb").read()).hexdigest()
    print(f"{got}  {name}")
    if got != want:
        bad.append(name)
print("\nвсе SHA-256 совпадают с замороженными" if not bad else f"\nРАСХОЖДЕНИЕ: {bad}")
mem = int(open("/proc/meminfo").read().split("MemTotal:")[1].split()[0]) / 1048576
print(f"ядер: {os.cpu_count()}, память: {mem:.0f} ГБ, Python {sys.version.split()[0]}")

In [ ]:
# 2. Пакеты закреплённых версий и Google Диск
rc = sh("pip install -q -r requirements.txt 2>&1 | tail -3")
sh("python3 -I -c 'import numpy, scipy, ripser, radicalpy, threadpoolctl; "
   "print(\"numpy\", numpy.__version__, \"scipy\", scipy.__version__)'")
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
print("Диск подключён:", DRIVE)

In [ ]:
# 3. Проверка окружения: smoke, физика, калибровка C3-C7 и контрольные числа на калибровочных птицах
t0 = time.time()
c = f"{WORK}/check"
rc = sh(f"python3 -I mag_s3.py smoke --out {c}")
rc = rc or sh(f"python3 -I mag_s3.py physics --out {c} > {c}/physics.log 2>&1; r=$?; tail -2 {c}/physics.log; exit $r")
rc = rc or sh(f"python3 -I mag_s3.py calibrate --out {c} --workers {os.cpu_count()}")
rc = rc or sh(f"python3 -I scripts/env_check.py {c} > {c}/env_check.out 2>&1; r=$?; tail -40 {c}/env_check.out; exit $r")
if not rc:
    sh(f"cp {c}/env_check.json {c}/calibration.json {DRIVE}/")
print(f"\nкод выхода {rc}, {(time.time() - t0) / 60:.1f} мин; файлы env_check.json и calibration.json в {DRIVE}")

In [ ]:
# 4. Слепая проверка конвейера на птицах 9301-9302 (выход удаляется, вердикты не печатаются)
t0 = time.time()
rc = sh(f"set -o pipefail; bash scripts/pipeline.sh blind {WORK} 12 2>&1 | tee {DRIVE}/blind_check.log")
print(f"\nкод выхода {rc}, {(time.time() - t0) / 60:.1f} мин; журнал: {DRIVE}/blind_check.log")

## Подтверждающий прогон

Запускать **только после заморозки протокола и кода и команды «запускай»**.
Ячейка считает частями по полчаса и после каждой части кладёт архив `MAG-S3/confirmatory_v04.tar.gz` на Диск.
Если сессия оборвётся, запусти ячейки 1, 2 и 5 снова: счёт продолжится из архива.

In [ ]:
# 5. Подтверждающий прогон
import shutil
t0 = time.time()
arch = f"{DRIVE}/confirmatory_v04.tar.gz"
if os.path.exists(arch) and not os.path.exists(f"{WORK}/confirmatory_v04"):
    sh(f"tar -xzf {arch} -C {WORK}")
    print("продолжаю из архива на Диске")
while True:
    rc = sh(f"set -o pipefail; bash scripts/pipeline.sh confirmatory {WORK} 0.5 2>&1 | tee -a {WORK}/confirmatory.log")
    sh(f"tar -czf {WORK}/part.tar.gz -C {WORK} confirmatory_v04 && cp {WORK}/part.tar.gz {arch}.tmp && "
       f"mv {arch}.tmp {arch}")
    print(f"архив обновлён, {(time.time() - t0) / 60:.0f} мин с начала")
    if rc != 3:
        break
sh(f"cp {WORK}/confirmatory.log {DRIVE}/; sha256sum {arch}")
print(f"\nкод выхода {rc}; итог в {WORK}/confirmatory_v04/final.log")
sh(f"cat {WORK}/confirmatory_v04/final.log")